In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import seaborn as sns
from tqdm import tqdm
%matplotlib inline

In [ ]:
# Task 1: Write your code here:

# Read the dataset Q1_data.csv using read_csv()

# Load the dataset
restaurant = os.path.join(path, 'Q1_data.csv')
df_restaurant = pd.read_csv(restaurant)

In [ ]:
# Task 2: Write your code here:

# Inspect the first few rows using head()

print(f"Dataset shape: {df_restaurant.shape}")
df_restaurant.head()

In [ ]:
# Task 3: Write your code here:

# Display dataset information using info()

# Check data types and structure
df_restaurant.info()

In [ ]:
# Task 4: Write your code here:

# Show statistical description using describe()

# Descriptive statistics for numerical columns
df_restaurant.describe()

In [ ]:
# Task 5: Write your code here:

# Plot the target distribution (delivery_time)

# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_restaurant['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

#Drop the 'Order_ID' column from the data
df = df_restaurant.drop(columns=['Order_ID'])

df.info()

In [ ]:
# Task 2: Write your code here:

# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Fill categorical columns with with mode
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
  df[col]=df[col].fillna(df[col].mode()[0])

In [ ]:
# Do we have missing values? after fillin the categorical columns with with mode?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Fill numerical features columns with with mean
for col in ['Courier_Experience_yrs', 'Delivery_Time']:
  df[col]=df[col].fillna(df[col].mean())

  print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:

# Check and remove duplicates if any exist

# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

# Encode categorical variables if needed (Bonus if used One Hot Encoding)

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
df.head(10)

In [ ]:
# Task 5: Write your code here:

# Apply feature scaling for all features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # We dont't scale our the target (Delivery_Time)

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:

# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

# we have imbalance target

In [ ]:
df

In [ ]:
# Task 1: Write your code here:

# Split the dataset into features (X) and target (y)

# Define features (X) and target (y)
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df[feature_cols]
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:


# Use the correct split: KFold OR StratifiedKFold
#Train a RandomForest model
# Evaluate using MAE (Mean Absolute Error) ONLY
# Print the averaged score across all folds

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

In [ ]:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "LightGBM": LGBMRegressor(verbose=-1),
}

In [ ]:
# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    if model_name == "Random Forest Regressor" :
      print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["MSA"].append(rmse) # Here is the MSA
    all_results[model_name]["r2"].append(r2)

In [ ]:
for model_name in all_results:
      if model_name == "Random Forest Regressor" :

        print(f"\n{model_name}:")
        print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
        print(f"  MSA: {np.mean(all_results[model_name]['rmse']):.4f}")
        print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Task 2: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:l